In [1]:
from pathlib import Path
import pandas as pd

# Find project root
if (Path.cwd() / "data" / "raw" / "netflix_titles.csv").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "data" / "raw" / "netflix_titles.csv").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Project root could not be located.")

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "netflix_titles.csv"
PROCESSED_FILE = PROJECT_ROOT / "data" / "processed" / "netflix_titles_cleaned.csv"

print("Project Root:", PROJECT_ROOT)
print("Raw Dataset:", RAW_FILE)
print("Processed Dataset:", PROCESSED_FILE)

Project Root: c:\Users\Shashiranjan kumar\OneDrive\Desktop\SWYNEX-Data-Preparation
Raw Dataset: c:\Users\Shashiranjan kumar\OneDrive\Desktop\SWYNEX-Data-Preparation\data\raw\netflix_titles.csv
Processed Dataset: c:\Users\Shashiranjan kumar\OneDrive\Desktop\SWYNEX-Data-Preparation\data\processed\netflix_titles_cleaned.csv


In [2]:
# Load the raw Netflix dataset

df = pd.read_csv(
    RAW_FILE,
    low_memory=False
)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully.
Rows: 8807
Columns: 12


In [3]:
# Inspect the dataset structure

print("Column Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Rows:")
display(df.head())

Column Names:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']

Data Types:
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

First 5 Rows:


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
# Missing value audit

missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage.round(2)
})

missing_report = missing_report[
    missing_report["Missing Values"] > 0
].sort_values(
    by="Missing Values",
    ascending=False
)

print("Missing Value Report")
display(missing_report)

Missing Value Report


,Missing Values,Missing Percentage
director,2634,29.91
country,831,9.44
cast,825,9.37
date_added,10,0.11
rating,4,0.05
duration,3,0.03


In [5]:
# Check for duplicate records

duplicate_rows = df.duplicated().sum()

print("Duplicate row count:", duplicate_rows)

if duplicate_rows == 0:
    print("Result: No duplicate rows found.")
else:
    print("Result: Duplicate rows found.")

Duplicate row count: 0
Result: No duplicate rows found.


In [6]:
# Validate show_id uniqueness

duplicate_show_ids = df["show_id"].duplicated().sum()

print("Duplicate show IDs:", duplicate_show_ids)

if duplicate_show_ids == 0:
    print("Result: All show IDs are unique.")
else:
    print("Result: Duplicate show IDs found.")

Duplicate show IDs: 0
Result: All show IDs are unique.


In [7]:
# Convert date_added to datetime

df["date_added"] = pd.to_datetime(
    df["date_added"],
    errors="coerce"
)

print("date_added data type:", df["date_added"].dtype)
print("Missing/invalid dates after conversion:", df["date_added"].isna().sum())

date_added data type: datetime64[ns]
Missing/invalid dates after conversion: 98


In [8]:
# Investigate date_added values that could not be converted

date_check = pd.to_datetime(
    df["date_added"],
    errors="coerce"
)

invalid_date_mask = (
    df["date_added"].notna()
    & date_check.isna()
)

print("Original missing dates:", df["date_added"].isna().sum())
print("Additional conversion failures:", invalid_date_mask.sum())
print("Total missing after conversion:", date_check.isna().sum())

print("\nExamples of values that failed conversion:")
display(
    df.loc[invalid_date_mask, ["show_id", "title", "date_added"]].head(20)
)

Original missing dates: 98
Additional conversion failures: 0
Total missing after conversion: 98

Examples of values that failed conversion:


,show_id,title,date_added


In [9]:
# Re-check date_added using the original raw dataset

raw_df = pd.read_csv(
    RAW_FILE,
    low_memory=False
)

raw_missing_dates = raw_df["date_added"].isna().sum()

print("Original raw date_added missing values:", raw_missing_dates)

Original raw date_added missing values: 10


In [10]:
# Handle missing values in categorical/text columns

missing_fill_columns = [
    "director",
    "cast",
    "country",
    "rating",
    "duration"
]

for column in missing_fill_columns:
    missing_count = df[column].isna().sum()

    df[column] = df[column].fillna("Not Specified")

    print(
        f"{column}: {missing_count} missing values "
        "replaced with 'Not Specified'"
    )

director: 2634 missing values replaced with 'Not Specified'
cast: 825 missing values replaced with 'Not Specified'
country: 831 missing values replaced with 'Not Specified'
rating: 4 missing values replaced with 'Not Specified'
duration: 3 missing values replaced with 'Not Specified'


In [11]:
# Safely convert date_added using the original raw data

raw_df = pd.read_csv(RAW_FILE, low_memory=False)

raw_df["date_added_clean"] = pd.to_datetime(
    raw_df["date_added"],
    errors="coerce",
    format="mixed"
)

print("Original raw missing dates:", raw_df["date_added"].isna().sum())
print(
    "Missing dates after conversion:",
    raw_df["date_added_clean"].isna().sum()
)
print(
    "Successfully converted dates:",
    raw_df["date_added_clean"].notna().sum()
)

Original raw missing dates: 10
Missing dates after conversion: 10
Successfully converted dates: 8797


In [12]:
# Convert date_added safely in the working dataset

df["date_added"] = pd.to_datetime(
    df["date_added"],
    errors="coerce",
    format="mixed"
)

print("date_added data type:", df["date_added"].dtype)
print("Missing date_added values:", df["date_added"].isna().sum())
print("Valid date_added values:", df["date_added"].notna().sum())

date_added data type: datetime64[ns]
Missing date_added values: 98
Valid date_added values: 8709


In [5]:
import pandas as pd
from pathlib import Path

# Project path
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "netflix_titles.csv"

# Load dataset
df = pd.read_csv(RAW_FILE, low_memory=False)

# Create working copy
cleaned_df = df.copy()

# Convert release_year to numeric
cleaned_df["release_year"] = pd.to_numeric(
    cleaned_df["release_year"],
    errors="coerce"
)

print("Dataset loaded successfully.")
print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))

Dataset loaded successfully.
Rows: 8807
Columns: 12


In [6]:
# Validate release_year

print("Minimum release year:", cleaned_df["release_year"].min())
print("Maximum release year:", cleaned_df["release_year"].max())

invalid_years = cleaned_df[
    (cleaned_df["release_year"] < 1900) |
    (cleaned_df["release_year"] > 2026)
]

print("Invalid release year count:", len(invalid_years))

if len(invalid_years) == 0:
    print("Result: No invalid release years found.")
else:
    print(invalid_years[["show_id", "title", "release_year"]].head(10))

Minimum release year: 1925
Maximum release year: 2021
Invalid release year count: 0
Result: No invalid release years found.


In [7]:
# Validate duration

print("Missing duration values:", cleaned_df["duration"].isna().sum())

print("\nUnique duration examples:")
print(cleaned_df["duration"].dropna().unique()[:20])

print("\nDuration data type:", cleaned_df["duration"].dtype)

Missing duration values: 3

Unique duration examples:
['90 min' '2 Seasons' '1 Season' '91 min' '125 min' '9 Seasons' '104 min'
 '127 min' '4 Seasons' '67 min' '94 min' '5 Seasons' '161 min' '61 min'
 '166 min' '147 min' '103 min' '97 min' '106 min' '111 min']

Duration data type: object


In [8]:
# Check missing duration records

missing_duration = cleaned_df[cleaned_df["duration"].isna()]

print("Missing duration records:")
print(missing_duration[["show_id", "title", "type", "duration"]])

Missing duration records:
     show_id                                 title   type duration
5541   s5542                       Louis C.K. 2017  Movie      NaN
5794   s5795                 Louis C.K.: Hilarious  Movie      NaN
5813   s5814  Louis C.K.: Live at the Comedy Store  Movie      NaN


In [9]:
# Apply final missing-value handling

cleaned_df["director"] = cleaned_df["director"].fillna("Not Specified")
cleaned_df["cast"] = cleaned_df["cast"].fillna("Not Specified")
cleaned_df["country"] = cleaned_df["country"].fillna("Not Specified")
cleaned_df["rating"] = cleaned_df["rating"].fillna("Not Specified")
cleaned_df["duration"] = cleaned_df["duration"].fillna("Not Specified")

# Convert date_added to datetime
cleaned_df["date_added"] = pd.to_datetime(
    cleaned_df["date_added"],
    errors="coerce"
)

print("Final cleaning applied successfully.")

Final cleaning applied successfully.


In [10]:
# Final missing-value validation

missing_report = cleaned_df.isnull().sum()

print("Final missing-value report:")
print(missing_report[missing_report > 0])

print("\nTotal missing values:", cleaned_df.isnull().sum().sum())

Final missing-value report:
date_added    98
dtype: int64

Total missing values: 98


In [11]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_FILE = PROJECT_ROOT / "data" / "processed" / "netflix_titles_cleaned.csv"

cleaned_df.to_csv(PROCESSED_FILE, index=False)

print("Cleaned dataset saved successfully.")
print("File:", PROCESSED_FILE)
print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))

Cleaned dataset saved successfully.
File: c:\Users\Shashiranjan kumar\OneDrive\Desktop\SWYNEX-Data-Preparation\data\processed\netflix_titles_cleaned.csv
Rows: 8807
Columns: 12


In [12]:
# Verify saved cleaned dataset

check_df = pd.read_csv(PROCESSED_FILE)

print("Saved file verified successfully.")
print("Rows:", check_df.shape[0])
print("Columns:", check_df.shape[1])

print("\nColumns:")
print(list(check_df.columns))

print("\nRemaining missing values:")
print(check_df.isnull().sum()[check_df.isnull().sum() > 0])

Saved file verified successfully.
Rows: 8807
Columns: 12

Columns:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']

Remaining missing values:
date_added    98
dtype: int64
